# 🚀 Hummingbot Analytics Pro - Jupyter Version
Профессиональный технический анализ криптовалютных пар

In [2]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import requests
import numpy as np
from datetime import datetime, timedelta
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'ipywidgets'

In [ ]:
class MarketDataProvider:
    """Провайдер рыночных данных с подключением к биржам"""
    
    @staticmethod
    def fetch_binance_data(symbol, interval='1d', limit=500):
        """Получение данных с Binance"""
        try:
            url = f"https://api.binance.com/api/v3/klines?symbol={symbol}&interval={interval}&limit={limit}"
            response = requests.get(url)
            data = response.json()
            
            df = pd.DataFrame(data, columns=[
                'timestamp', 'open', 'high', 'low', 'close', 'volume',
                'close_time', 'quote_asset_volume', 'trades',
                'taker_buy_base', 'taker_buy_quote', 'ignore'
            ])
            
            df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
            for col in ['open', 'high', 'low', 'close', 'volume']:
                df[col] = df[col].astype(float)
            
            return df
        
        except Exception as e:
            print(f"Ошибка Binance API: {str(e)}")
            return None

    @staticmethod
    def fetch_binance_ticker(symbol):
        """Получение текущих цен с Binance"""
        try:
            url = f"https://api.binance.com/api/v3/ticker/24hr?symbol={symbol}"
            response = requests.get(url)
            return response.json()
        except:
            return None

    @staticmethod
    def fetch_order_book(symbol, limit=20):
        """Получение стакана цен"""
        try:
            url = f"https://api.binance.com/api/v3/depth?symbol={symbol}&limit={limit}"
            response = requests.get(url)
            return response.json()
        except:
            return None

In [ ]:
class TechnicalAnalysis:
    """Технический анализ с индикаторами (ручная реализация)"""
    
    @staticmethod
    def calculate_all_indicators(df):
        """Расчет всех технических индикаторов"""
        if df is None or df.empty:
            return df
            
        try:
            # RSI
            df['rsi_14'] = TechnicalAnalysis.calculate_rsi(df['close'], 14)
            df['rsi_7'] = TechnicalAnalysis.calculate_rsi(df['close'], 7)
            
            # MACD
            macd_data = TechnicalAnalysis.calculate_macd(df['close'])
            df['macd'] = macd_data['macd']
            df['macd_signal'] = macd_data['signal']
            df['macd_histogram'] = macd_data['histogram']
            
            # Bollinger Bands
            bb_data = TechnicalAnalysis.calculate_bollinger_bands(df['close'])
            df['bb_upper'] = bb_data['upper']
            df['bb_middle'] = bb_data['middle']
            df['bb_lower'] = bb_data['lower']
            df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / df['bb_middle']
            
            # SuperTrend
            df['supertrend'] = TechnicalAnalysis.calculate_supertrend(df)
            
            # Moving Averages
            df['sma_20'] = TechnicalAnalysis.calculate_sma(df['close'], 20)
            df['sma_50'] = TechnicalAnalysis.calculate_sma(df['close'], 50)
            df['ema_12'] = TechnicalAnalysis.calculate_ema(df['close'], 12)
            df['ema_26'] = TechnicalAnalysis.calculate_ema(df['close'], 26)
            df['ema_50'] = TechnicalAnalysis.calculate_ema(df['close'], 50)
            
            # Volume indicators
            df['volume_sma'] = TechnicalAnalysis.calculate_sma(df['volume'], 20)
            df['volume_ratio'] = df['volume'] / df['volume_sma']
            df['obv'] = TechnicalAnalysis.calculate_obv(df['close'], df['volume'])
            
            # Volatility
            df['atr'] = TechnicalAnalysis.calculate_atr(df)
            df['atr_percent'] = (df['atr'] / df['close']) * 100
            
            # Support and Resistance
            df = TechnicalAnalysis.calculate_pivot_points(df)
            df = TechnicalAnalysis.calculate_fibonacci_levels(df)
            
            # Momentum indicators
            df['stoch_rsi'] = TechnicalAnalysis.calculate_stoch_rsi(df['close'])
            df['williams_r'] = TechnicalAnalysis.calculate_williams_r(df['high'], df['low'], df['close'])
            df['cci'] = TechnicalAnalysis.calculate_cci(df['high'], df['low'], df['close'])
            
            # Trend indicators
            df['adx'] = TechnicalAnalysis.calculate_adx(df['high'], df['low'], df['close'])
            
            return df
            
        except Exception as e:
            print(f"Ошибка расчета индикаторов: {str(e)}")
            return df

    @staticmethod
    def calculate_rsi(prices, period=14):
        """Расчет RSI"""
        delta = prices.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        return rsi

    @staticmethod
    def calculate_macd(prices, fast=12, slow=26, signal=9):
        """Расчет MACD"""
        ema_fast = prices.ewm(span=fast).mean()
        ema_slow = prices.ewm(span=slow).mean()
        macd = ema_fast - ema_slow
        macd_signal = macd.ewm(span=signal).mean()
        macd_histogram = macd - macd_signal
        
        return {'macd': macd, 'signal': macd_signal, 'histogram': macd_histogram}

    @staticmethod
    def calculate_bollinger_bands(prices, period=20, std=2):
        """Расчет полос Боллинджера"""
        sma = prices.rolling(window=period).mean()
        std_dev = prices.rolling(window=period).std()
        
        upper_band = sma + (std_dev * std)
        lower_band = sma - (std_dev * std)
        
        return {'upper': upper_band, 'middle': sma, 'lower': lower_band}

    @staticmethod
    def calculate_supertrend(df, period=10, multiplier=3):
        """Расчет SuperTrend"""
        hl2 = (df['high'] + df['low']) / 2
        atr = TechnicalAnalysis.calculate_atr(df, period)
        
        upper_band = hl2 + (multiplier * atr)
        lower_band = hl2 - (multiplier * atr)
        
        supertrend = pd.Series(index=df.index, dtype=float)
        
        for i in range(1, len(df)):
            if df['close'].iloc[i] > upper_band.iloc[i-1]:
                supertrend.iloc[i] = lower_band.iloc[i]
            elif df['close'].iloc[i] < lower_band.iloc[i-1]:
                supertrend.iloc[i] = upper_band.iloc[i]
            else:
                supertrend.iloc[i] = supertrend.iloc[i-1]
                
        return supertrend

    @staticmethod
    def calculate_sma(prices, period):
        """Простое скользящее среднее"""
        return prices.rolling(window=period).mean()

    @staticmethod
    def calculate_ema(prices, period):
        """Экспоненциальное скользящее среднее"""
        return prices.ewm(span=period).mean()

    @staticmethod
    def calculate_obv(close, volume):
        """On-Balance Volume"""
        obv = [0]
        for i in range(1, len(close)):
            if close.iloc[i] > close.iloc[i-1]:
                obv.append(obv[-1] + volume.iloc[i])
            elif close.iloc[i] < close.iloc[i-1]:
                obv.append(obv[-1] - volume.iloc[i])
            else:
                obv.append(obv[-1])
        return pd.Series(obv, index=close.index)

    @staticmethod
    def calculate_atr(df, period=14):
        """Average True Range"""
        high_low = df['high'] - df['low']
        high_close = abs(df['high'] - df['close'].shift())
        low_close = abs(df['low'] - df['close'].shift())
        
        true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
        atr = true_range.rolling(window=period).mean()
        return atr

    @staticmethod
    def calculate_pivot_points(df):
        """Точки разворота"""
        df['pivot'] = (df['high'] + df['low'] + df['close']) / 3
        df['r1'] = 2 * df['pivot'] - df['low']
        df['s1'] = 2 * df['pivot'] - df['high']
        df['r2'] = df['pivot'] + (df['high'] - df['low'])
        df['s2'] = df['pivot'] - (df['high'] - df['low'])
        return df

    @staticmethod
    def calculate_fibonacci_levels(df, lookback=50):
        """Уровни Фибоначчи"""
        high = df['high'].rolling(lookback).max()
        low = df['low'].rolling(lookback).min()
        
        df['fib_0'] = high
        df['fib_0.236'] = high - (high - low) * 0.236
        df['fib_0.382'] = high - (high - low) * 0.382
        df['fib_0.5'] = high - (high - low) * 0.5
        df['fib_0.618'] = high - (high - low) * 0.618
        df['fib_0.786'] = high - (high - low) * 0.786
        df['fib_1'] = low
        
        return df

    @staticmethod
    def calculate_stoch_rsi(close, period=14, k_period=3, d_period=3):
        """Stochastic RSI"""
        rsi = TechnicalAnalysis.calculate_rsi(close, period)
        stoch_rsi = (rsi - rsi.rolling(period).min()) / (rsi.rolling(period).max() - rsi.rolling(period).min())
        return stoch_rsi.rolling(k_period).mean()

    @staticmethod
    def calculate_williams_r(high, low, close, period=14):
        """Williams %R"""
        highest_high = high.rolling(period).max()
        lowest_low = low.rolling(period).min()
        williams_r = (highest_high - close) / (highest_high - lowest_low) * -100
        return williams_r

    @staticmethod
    def calculate_cci(high, low, close, period=20):
        """Commodity Channel Index"""
        typical_price = (high + low + close) / 3
        sma = typical_price.rolling(period).mean()
        mad = typical_price.rolling(period).apply(lambda x: np.mean(np.abs(x - np.mean(x))))
        cci = (typical_price - sma) / (0.015 * mad)
        return cci

    @staticmethod
    def calculate_adx(high, low, close, period=14):
        """Average Directional Index (упрощенная версия)"""
        # Упрощенный расчет ADX
        tr = TechnicalAnalysis.calculate_atr(pd.DataFrame({'high': high, 'low': low, 'close': close}))
        plus_dm = high.diff()
        minus_dm = low.diff().abs()
        
        plus_di = 100 * (plus_dm.rolling(period).mean() / tr)
        minus_di = 100 * (minus_dm.rolling(period).mean() / tr)
        
        dx = 100 * abs(plus_di - minus_di) / (plus_di + minus_di)
        adx = dx.rolling(period).mean()
        return adx

In [ ]:
def get_custom_layout():
    """Кастомный layout для графиков"""
    return {
        'margin': dict(l=50, r=50, t=80, b=50),
        'plot_bgcolor': '#131722',
        'paper_bgcolor': '#131722',
        'font': dict(color='#d1d4dc'),
        'xaxis': dict(gridcolor='#1c2030'),
        'yaxis': dict(gridcolor='#1c2030'),
        'showlegend': True,
    }

def show_market_overview(symbol, data):
    """Обзор рынка для выбранной пары"""
    print("## 📊 Обзор рынка")
    
    if data is not None and not data.empty:
        current = data.iloc[-1]
        ticker = MarketDataProvider.fetch_binance_ticker(symbol)
        
        if ticker:
            price_change = float(ticker['priceChangePercent'])
            volume = float(ticker['volume'])
            high = float(ticker['highPrice'])
            low = float(ticker['lowPrice'])
            range_pct = ((high - low) / low) * 100
            trades = int(ticker['count'])
            quote_volume = float(ticker['quoteVolume'])
            price_change_amt = float(ticker['priceChange'])
            
            overview_data = {
                'Метрика': ['Цена', '24ч Объем', 'Диапазон 24ч', 'Сделки 24ч', 'Объем USDT', 'Изменение цены'],
                'Значение': [
                    f"${float(ticker['lastPrice']):.4f}",
                    f"${volume:,.0f}",
                    f"{range_pct:.2f}%",
                    f"{trades:,}",
                    f"${quote_volume:,.0f}",
                    f"${price_change_amt:.4f}"
                ],
                'Изменение': [
                    f"{price_change:.2f}%",
                    "-", "-", "-", "-", "-"
                ]
            }
            
            overview_df = pd.DataFrame(overview_data)
            display(overview_df)

def show_technical_metrics(data):
    """Технические метрики"""
    if data is None or data.empty:
        return
    
    current = data.iloc[-1]
    
    print("## 🔧 Технические индикаторы")
    
    # Основные осцилляторы
    rsi = current.get('rsi_14', 50)
    rsi_status = "Перепродан" if rsi < 30 else "Перекуплен" if rsi > 70 else "Нейтрален"
    
    macd = current.get('macd', 0)
    signal = current.get('macd_signal', 0)
    macd_signal = "Бычий" if macd > signal else "Медвежий"
    
    stoch_rsi = current.get('stoch_rsi', 0.5)
    stoch_signal = "Перепродан" if stoch_rsi < 0.2 else "Перекуплен" if stoch_rsi > 0.8 else "Нейтрален"
    
    williams_r = current.get('williams_r', -50)
    williams_signal = "Перепродан" if williams_r < -80 else "Перекуплен" if williams_r > -20 else "Нейтрален"
    
    # Тренд и волатильность
    adx = current.get('adx', 25)
    trend_strength = "Сильный" if adx > 25 else "Слабый" if adx < 20 else "Умеренный"
    
    atr_percent = current.get('atr_percent', 0)
    
    bb_width = current.get('bb_width', 0) * 100
    bb_squeeze = "Сжатие" if bb_width < 2 else "Расширение" if bb_width > 5 else "Норма"
    
    cci = current.get('cci', 0)
    cci_signal = "Перепродан" if cci < -100 else "Перекуплен" if cci > 100 else "Нейтрален"
    
    tech_data = {
        'Индикатор': ['RSI (14)', 'MACD', 'Stoch RSI', 'Williams %R', 'ADX', 'ATR', 'BB Width', 'CCI'],
        'Значение': [f"{rsi:.1f}", macd_signal, f"{stoch_rsi:.2f}", f"{williams_r:.1f}", 
                   f"{adx:.1f}", f"{atr_percent:.2f}%", f"{bb_width:.2f}%", f"{cci:.1f}"],
        'Сигнал': [rsi_status, "-", stoch_signal, williams_signal, trend_strength, 
                  "Волатильность", bb_squeeze, cci_signal]
    }
    
    tech_df = pd.DataFrame(tech_data)
    display(tech_df)

In [ ]:
def show_market_sentiment(data):
    """Анализ настроений рынка с графиками"""
    if data is None or data.empty:
        return
    
    print("## 🎯 Анализ настроений")
    
    # Создаем графики настроений
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('RSI - Сила тренда', 'MACD - Моментум', 'Объемы', 'Волатильность ATR'),
        vertical_spacing=0.1
    )
    
    # RSI
    fig.add_trace(go.Scatter(
        x=data['timestamp'], y=data['rsi_14'],
        line=dict(color='purple'), name='RSI'
    ), row=1, col=1)
    
    # Добавляем зоны перекупленности/перепроданности
    fig.add_hrect(y0=70, y1=100, line_width=0, fillcolor="red", opacity=0.1, row=1, col=1)
    fig.add_hrect(y0=0, y1=30, line_width=0, fillcolor="green", opacity=0.1, row=1, col=1)
    
    # MACD
    fig.add_trace(go.Scatter(
        x=data['timestamp'], y=data['macd'],
        line=dict(color='blue'), name='MACD'
    ), row=1, col=2)
    fig.add_trace(go.Scatter(
        x=data['timestamp'], y=data['macd_signal'],
        line=dict(color='red'), name='Signal'
    ), row=1, col=2)
    
    # Объемы
    colors = ['green' if close >= open else 'red' 
             for close, open in zip(data['close'], data['open'])]
    fig.add_trace(go.Bar(
        x=data['timestamp'], y=data['volume'],
        marker_color=colors, name='Volume'
    ), row=2, col=1)
    
    # ATR
    fig.add_trace(go.Scatter(
        x=data['timestamp'], y=data['atr_percent'],
        line=dict(color='orange'), name='ATR %'
    ), row=2, col=2)
    
    fig.update_layout(height=600, **get_custom_layout())
    fig.show()
    
    # Дополнительные метрики настроений
    current = data.iloc[-1]
    
    # Сила тренда
    adx = current.get('adx', 25)
    if adx > 25:
        trend_strength = "💪 Сильный"
    elif adx > 20:
        trend_strength = "👊 Умеренный"
    else:
        trend_strength = "🫣 Слабый"
    
    # Направление тренда
    ema_12 = current.get('ema_12', 0)
    ema_26 = current.get('ema_26', 0)
    trend = "📈 Бычий" if ema_12 > ema_26 else "📉 Медвежий"
    
    # Волатильность
    atr_percent = current.get('atr_percent', 0)
    if atr_percent > 3:
        vol_status = "🌪️ Высокая"
    elif atr_percent > 1.5:
        vol_status = "🌊 Средняя"
    else:
        vol_status = "🌿 Низкая"
    
    # Объемы
    volume_ratio = current.get('volume_ratio', 1)
    if volume_ratio > 2:
        vol_activity = "🔥 Аномальный"
    elif volume_ratio > 1.2:
        vol_activity = "⚡ Высокий"
    else:
        vol_activity = "💤 Нормальный"
    
    sentiment_data = {
        'Метрика': ['Сила тренда', 'Тренд EMA', 'Волатильность', 'Активность объемов'],
        'Статус': [trend_strength, trend, vol_status, vol_activity]
    }
    
    sentiment_df = pd.DataFrame(sentiment_data)
    display(sentiment_df)

In [ ]:
def show_price_chart_with_indicators(data, symbol):
    """Главный график с индикаторами"""
    print("## 📈 Основной график")
    
    # Создание графика
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        row_heights=[0.7, 0.3],
        subplot_titles=('Цена', 'Объем')
    )
    
    # Свечи
    fig.add_trace(go.Candlestick(
        x=data['timestamp'],
        open=data['open'],
        high=data['high'],
        low=data['low'],
        close=data['close'],
        name='Цена'
    ), row=1, col=1)
    
    # Индикаторы
    # Полосы Боллинджера
    fig.add_trace(go.Scatter(
        x=data['timestamp'], y=data['bb_upper'],
        line=dict(color='rgba(250, 250, 250, 0.5)', width=1),
        name='BB Верхняя'
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=data['timestamp'], y=data['bb_lower'],
        line=dict(color='rgba(250, 250, 250, 0.5)', width=1),
        fill='tonexty',
        fillcolor='rgba(250, 250, 250, 0.1)',
        name='BB Нижняя'
    ), row=1, col=1)
    
    # SuperTrend
    fig.add_trace(go.Scatter(
        x=data['timestamp'], y=data['supertrend'],
        line=dict(color='yellow', width=2),
        name='SuperTrend'
    ), row=1, col=1)
    
    # Скользящие средние
    for ma in ['sma_20', 'sma_50', 'ema_12', 'ema_26']:
        if ma in data.columns:
            fig.add_trace(go.Scatter(
                x=data['timestamp'], y=data[ma],
                line=dict(width=1), name=ma.upper()
            ), row=1, col=1)
    
    # Объемы
    colors = ['green' if close >= open else 'red' 
             for close, open in zip(data['close'], data['open'])]
    fig.add_trace(go.Bar(
        x=data['timestamp'], y=data['volume'],
        marker_color=colors, name='Объем'
    ), row=2, col=1)
    
    fig.update_layout(
        title=f'{symbol} - Технический анализ',
        height=800,
        **get_custom_layout()
    )
    
    fig.update_xaxes(rangeslider_visible=False)
    fig.show()

In [ ]:
def show_advanced_analysis(data):
    """Расширенный анализ"""
    print("## 🔬 Расширенный анализ")
    
    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=('MACD', 'Stochastic RSI', 'Williams %R', 'CCI', 'ADX', 'OBV'),
        vertical_spacing=0.08
    )
    
    # MACD
    fig.add_trace(go.Scatter(
        x=data['timestamp'], y=data['macd'],
        line=dict(color='blue'), name='MACD'
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=data['timestamp'], y=data['macd_signal'],
        line=dict(color='red'), name='Signal'
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        x=data['timestamp'], y=data['macd_histogram'],
        marker_color='gray', name='Histogram'
    ), row=1, col=1)
    
    # Stochastic RSI
    fig.add_trace(go.Scatter(
        x=data['timestamp'], y=data['stoch_rsi'],
        line=dict(color='purple'), name='Stoch RSI'
    ), row=1, col=2)
    fig.add_hrect(y0=0.8, y1=1, line_width=0, fillcolor="red", opacity=0.1, row=1, col=2)
    fig.add_hrect(y0=0, y1=0.2, line_width=0, fillcolor="green", opacity=0.1, row=1, col=2)
    
    # Williams %R
    fig.add_trace(go.Scatter(
        x=data['timestamp'], y=data['williams_r'],
        line=dict(color='orange'), name='Williams %R'
    ), row=2, col=1)
    fig.add_hrect(y0=-20, y1=0, line_width=0, fillcolor="red", opacity=0.1, row=2, col=1)
    fig.add_hrect(y0=-100, y1=-80, line_width=0, fillcolor="green", opacity=0.1, row=2, col=1)
    
    # CCI
    fig.add_trace(go.Scatter(
        x=data['timestamp'], y=data['cci'],
        line=dict(color='cyan'), name='CCI'
    ), row=2, col=2)
    fig.add_hrect(y0=100, y1=200, line_width=0, fillcolor="red", opacity=0.1, row=2, col=2)
    fig.add_hrect(y0=-200, y1=-100, line_width=0, fillcolor="green", opacity=0.1, row=2, col=2)
    
    # ADX
    fig.add_trace(go.Scatter(
        x=data['timestamp'], y=data['adx'],
        line=dict(color='yellow'), name='ADX'
    ), row=3, col=1)
    
    # OBV
    fig.add_trace(go.Scatter(
        x=data['timestamp'], y=data['obv'],
        line=dict(color='white'), name='OBV'
    ), row=3, col=2)
    
    fig.update_layout(height=800, **get_custom_layout())
    fig.show()

In [ ]:
# Виджеты для интерфейса
symbol_input = widgets.Text(
    value='BTCUSDT',
    placeholder='Введите символ (например BTCUSDT)',
    description='Пара:',
    disabled=False
)

interval_dropdown = widgets.Dropdown(
    options=['1m', '3m', '5m', '15m', '30m', '1h', '2h', '4h', '6h', '8h', '12h', '1d', '3d', '1w', '1M'],
    value='1d',
    description='Интервал:',
    disabled=False,
)

analyze_button = widgets.Button(
    description='🚀 Анализировать',
    disabled=False,
    button_style='success',
    tooltip='Запустить анализ',
    icon='rocket'
)

output = widgets.Output()

def on_analyze_button_clicked(b):
    with output:
        clear_output()
        
        symbol = symbol_input.value.upper()
        interval = interval_dropdown.value
        
        print(f"# 🔍 Анализ {symbol} ({interval})")
        print("Загрузка данных...")
        
        # Получение данных
        data = MarketDataProvider.fetch_binance_data(symbol, interval)
        
        if data is None or data.empty:
            print("❌ Ошибка загрузки данных. Проверьте символ и интервал.")
            return
        
        print("📊 Расчет индикаторов...")
        data = TechnicalAnalysis.calculate_all_indicators(data)
        
        # Показать результаты
        show_market_overview(symbol, data)
        show_technical_metrics(data)
        show_market_sentiment(data)
        show_price_chart_with_indicators(data, symbol)
        show_advanced_analysis(data)
        
        print("✅ Анализ завершен!")

analyze_button.on_click(on_analyze_button_clicked)

# Отображение интерфейса
print("# 🚀 Hummingbot Analytics Pro - Jupyter Version")
print("Профессиональный технический анализ криптовалютных пар")
print("\n## ⚙️ Настройки анализа")

display(widgets.HBox([symbol_input, interval_dropdown, analyze_button]))
display(output)

In [ ]:
# Автоматический запуск анализа для примера
print("## 🎯 Пример анализа BTCUSDT")
print("Нажмите кнопку '🚀 Анализировать' выше или выполните ячейку ниже для автоматического анализа...")

In [ ]:
# Автоматический запуск (раскомментируйте для автоматического анализа)
# symbol = "BTCUSDT"
# interval = "1d"
# data = MarketDataProvider.fetch_binance_data(symbol, interval)
# if data is not None:
#     data = TechnicalAnalysis.calculate_all_indicators(data)
#     show_market_overview(symbol, data)
#     show_technical_metrics(data)
#     show_market_sentiment(data)
#     show_price_chart_with_indicators(data, symbol)
#     show_advanced_analysis(data)